# RAGAS Evaluation — RAG_Pipeline

This notebook measures retrieval and generation quality of `RAGPipeline` with
[RAGAS](https://docs.ragas.io). It computes four metrics:

| Metric | What it measures |
|---|---|
| **faithfulness** | Are the answer's claims supported by the retrieved context? |
| **answer_relevancy** | Does the answer actually address the question? |
| **context_precision** | Are the relevant chunks ranked highly in the retrieved set? |
| **context_recall** | Does the retrieved context contain everything needed for the ground truth? |

## Methodology
1. A fixed corpus (`evaluation/datasets/ragas_corpus.json`, 20 passages) is ingested into the pipeline.
2. A 22-question set (`evaluation/datasets/ragas_eval.jsonl`) is run through `RAGPipeline.query()` in synchronous mode.
3. For each question we record the generated answer, the retrieved contexts, the latency, and the pipeline's consistency score.
4. RAGAS judges each sample with an LLM and aggregates the four metrics.

## Environment notes (important)
- **Run this with Python 3.11.** RAGAS 0.2.x's async executor crashes on Python 3.14 (`asyncio.timeout` task-context change). The pinned LangChain stack is `langchain-core/community 0.3.x` + `langchain-openai 0.3.x` (ragas 0.2.15 imports `langchain_community.chat_models.vertexai`, removed in 0.4.x).
- The judge LLM is read from the same env the pipeline uses (`OPENAI_API_KEY` / `LLM_BASE_URL` / `LLM_MODEL`), so it works with OpenAI-compatible providers (e.g. Groq). Embeddings for the relevancy metric use the local `bge-large-en-v1.5` model, so no OpenAI embeddings endpoint is required.
- **Provider rate limits:** RAGAS makes many judge calls. On a free tier you may hit a daily token cap before the full run finishes (this happened on 2026-06-23 with Groq at ~28%). Use a paid/higher-quota key or re-run after reset.

## Limitations
- RAGAS metrics are LLM-judged and have run-to-run variance — treat them as estimates.
- The corpus is small and curated, so scores reflect a controlled setting, not arbitrary production documents.
- Latency depends on machine, model, provider queueing, and `evaluator_mode`.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (OpenAI or any OpenAI-compatible key, e.g. Groq)."
print("LLM model:", os.getenv("LLM_MODEL"))
print("LLM base url:", os.getenv("LLM_BASE_URL") or "(default OpenAI)")

In [ ]:
# The verified runner does ingest -> query(22) -> RAGAS scoring and writes a JSON report.
# Running it here keeps the notebook identical to `python -m evaluation.run_ragas`.
from pathlib import Path
from evaluation.run_ragas import run

report = run(
    corpus_path=Path("evaluation/datasets/ragas_corpus.json"),
    dataset_path=Path("evaluation/datasets/ragas_eval.jsonl"),
    output_path=Path("evaluation/reports/ragas-latest.json"),
    limit=None,  # set e.g. limit=3 for a quick smoke test within a tight token budget
)
report

In [ ]:
# Pretty-print the measured metrics + latency
import json
print(json.dumps(report["metrics"], indent=2))
print("latency_ms:", json.dumps(report["latency_ms"], indent=2))

## Findings

**Latest run — 2026-06-23 (Groq `llama-3.3-70b-versatile` judge, local `bge-large` embeddings):**

- Pipeline stage: **complete** for all 22 questions. Measured end-to-end latency
  (sync-evaluator mode): mean ≈ 9.3 s, median ≈ 9.5 s, p95 ≈ 12.5 s, range 2.6–14.2 s.
- RAGAS metrics: **NOT VERIFIED** — the run reached ~28% of judge calls before the Groq
  free-tier **daily token cap** (100k tokens/day; used 99,323) returned HTTP 429 for every
  subsequent call. Faithfulness / answer relevancy / context precision / recall are therefore
  not yet populated.

Full machine-readable record: [`evaluation/reports/ragas-run-2026-06-23.json`](evaluation/reports/ragas-run-2026-06-23.json).

**To complete the RAGAS metrics:** re-run this notebook (or `python -m evaluation.run_ragas`)
with available quota — e.g. after the daily reset, or with a paid/higher-quota OpenAI-compatible
key. The result is written to `evaluation/reports/ragas-latest.json`; copy the four metric values
into the README's Key Metrics section once they are real.